In [1]:
# Cell 1 - Setup
from dotenv import load_dotenv
from groq import Groq
import os

load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))
model = "llama-3.3-70b-versatile"

In [2]:
# Cell 2 - Helper functions
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

def chat(
        messages, 
        system=None, 
        temperature=1.0, 
        stop=None, 
        thinking=False, 
        thinking_budget=1024
    ):
    all_messages = []

    # If thinking mode enabled, inject system prompt to simulate it
    if thinking:
        thinking_system = """Before responding, think through the problem carefully.
Use <thinking> tags to show your reasoning process step by step.
Then provide your final answer after </thinking>.
Be thorough in your thinking - explore multiple angles before concluding."""
        all_messages.append({"role": "system", "content": thinking_system})
    elif system:
        all_messages.append({"role": "system", "content": system})

    all_messages += messages

    response = client.chat.completions.create(
        model=model,
        max_tokens=4000,
        messages=all_messages,
        temperature=temperature,
        stop=stop,
    )
    return response

def text_from_message(response):
    return response.choices[0].message.content or ""

def thinking_from_message(response):
    """Extract just the thinking part"""
    content = text_from_message(response)
    if "<thinking>" in content and "</thinking>" in content:
        start = content.index("<thinking>") + len("<thinking>")
        end = content.index("</thinking>")
        return content[start:end].strip()
    return "No thinking block found"

def answer_from_message(response):
    """Extract just the final answer (after thinking)"""
    content = text_from_message(response)
    if "</thinking>" in content:
        return content[content.index("</thinking>") + len("</thinking>"):].strip()
    return content

In [3]:
# Cell 3 - Basic thinking example
messages = []
add_user_message(messages, "What is 25 * 47 + 133? Show your work.")

response = chat(messages, thinking=True)
print("=== FULL RESPONSE ===")
print(text_from_message(response))

=== FULL RESPONSE ===
<thinking>
To solve this problem, we need to follow the order of operations (PEMDAS):
1. Multiply 25 and 47.
2. Add 133 to the result of the multiplication.

First, let's calculate the multiplication:
25 * 47 = 1175

Now, let's add 133 to the result:
1175 + 133 = 1308

So, the final result is the sum of the multiplication and 133.
</thinking>

The final answer is 1308.


In [4]:
# Cell 4 - Extract thinking vs answer separately
messages = []
add_user_message(messages, "Is 1997 a prime number? Think carefully.")

response = chat(messages, thinking=True)

print("=== THINKING PROCESS ===")
print(thinking_from_message(response))

print("\n=== FINAL ANSWER ===")
print(answer_from_message(response))

=== THINKING PROCESS ===
To determine if 1997 is a prime number, we need to consider its factors. A prime number is a number greater than 1 that has no positive divisors other than 1 and itself. 

First, let's recall that even numbers (except for 2) are not prime because they are divisible by 2. Since 1997 is an odd number, we can rule out 2 as a factor.

Next, we should try dividing 1997 by the smallest odd prime numbers: 3, 5, 7, 11, etc., and see if any of them divide evenly into 1997.

Checking divisibility by 3: The sum of the digits of 1997 is 1 + 9 + 9 + 7 = 26, which is not divisible by 3, so 1997 is not divisible by 3.

Checking divisibility by 5: Since 1997 does not end in 0 or 5, it is not divisible by 5.

Checking divisibility by 7: This requires actual division or a more complex rule. Upon division, we find that 1997 ÷ 7 = 285.285..., which is not a whole number, so 1997 is not divisible by 7.

Continuing this process for higher prime numbers up to the square root of 1997 

In [5]:
# Cell 5 - Complex reasoning example (like the course shows)
messages = []
add_user_message(messages, """
A bat and a ball together cost $1.10.
The bat costs $1.00 more than the ball.
How much does the ball cost?
Think through this carefully before answering.
""")

response = chat(messages, thinking=True)

print("=== THINKING PROCESS ===")
print(thinking_from_message(response))

print("\n=== FINAL ANSWER ===")
print(answer_from_message(response))

=== THINKING PROCESS ===
To solve this problem, let's assign a variable to the cost of the ball. Let's say the ball costs x cents. 
Since the bat costs $1.00 more than the ball, the bat costs x + $1.00.
The total cost of the bat and the ball together is $1.10, so we can set up an equation: x + (x + $1.00) = $1.10.
Combine like terms: 2x + $1.00 = $1.10.
Subtract $1.00 from both sides: 2x = $0.10.
Divide both sides by 2: x = $0.05.
So, the ball costs $0.05, or 5 cents.

=== FINAL ANSWER ===
The ball costs $0.05.


In [6]:
# Cell 6 - Multi-turn conversation with thinking
messages = []

# First turn
add_user_message(messages, "I have a list: [3, 1, 4, 1, 5, 9, 2, 6]. Sort it and find the median.")
response = chat(messages, thinking=True)

print("Turn 1 Thinking:")
print(thinking_from_message(response))
print("\nTurn 1 Answer:")
print(answer_from_message(response))

# Add to history and continue
add_assistant_message(messages, text_from_message(response))

# Second turn
add_user_message(messages, "Now find the mean of the same list.")
response2 = chat(messages, thinking=True)

print("\nTurn 2 Thinking:")
print(thinking_from_message(response2))
print("\nTurn 2 Answer:")
print(answer_from_message(response2))

Turn 1 Thinking:
To find the median of the list, I first need to sort the list in ascending order. 
The list is: [3, 1, 4, 1, 5, 9, 2, 6]. 
Sorting the list, I get: [1, 1, 2, 3, 4, 5, 6, 9]. 
Since the list has an even number of elements (8), the median will be the average of the two middle elements. 
The two middle elements are the 4th and 5th elements, which are 3 and 4. 
The average of 3 and 4 is (3 + 4) / 2 = 7 / 2 = 3.5.

Turn 1 Answer:
The median of the list is 3.5.

Turn 2 Thinking:
To find the mean of the list, I need to add up all the elements and then divide by the total number of elements.
The list is: [3, 1, 4, 1, 5, 9, 2, 6].
Let's add up the elements: 
3 + 1 = 4
4 + 4 = 8
8 + 1 = 9
9 + 5 = 14
14 + 9 = 23
23 + 2 = 25
25 + 6 = 31
The sum of the elements is 31.
The total number of elements is 8.
To find the mean, I divide the sum by the total number of elements: 31 / 8 = 3.875.

Turn 2 Answer:
The mean of the list is 3.875.


In [7]:
# Cell 7 - Thinking with higher budget (more detailed reasoning)
messages = []
add_user_message(messages, """
There are 3 boxes. One has only apples, one has only oranges, 
one has both. All boxes are labeled WRONG. 
You can pick one fruit from one box. 
Which box do you pick from to identify all boxes?
""")

# thinking_budget is just for reference here
# Groq doesn't have token budgets but higher temperature = more exploration
response = chat(messages, thinking=True, temperature=0.7)

print("=== THINKING ===")
print(thinking_from_message(response))
print("\n=== ANSWER ===")
print(answer_from_message(response))

=== THINKING ===
To identify all boxes, we need to gather information from the labels and the fruit we pick. 
Since all labels are wrong, the box labeled "apples" can't have only apples, the box labeled "oranges" can't have only oranges, and the box labeled "both" can't have both. 
Let's consider each option:
- If we pick from the box labeled "apples", and we get an apple, then this box must be the one with both (since the label "apples" is wrong), but if we get an orange, then this box must be the one with only oranges (since the label "apples" is wrong). 
- If we pick from the box labeled "oranges", and we get an orange, then this box must be the one with both (since the label "oranges" is wrong), but if we get an apple, then this box must be the one with only apples (since the label "oranges" is wrong).
- If we pick from the box labeled "both", and we get either an apple or an orange, then this box must be the one with only the fruit we picked (since the label "both" is wrong), beca